<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/VLM-Experiments/Classifier%2BVLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch
!pip install -q timm
!pip install -q evaluate
!pip install -q peft
!pip install --upgrade -q torchao
!pip install -q scikit-learn
!pip install -q matplotlib seaborn accelerate

In [ ]:
import os
import re
import gc
import time
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import CrossEntropyLoss

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from datasets import load_dataset, DatasetDict

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from torchvision.transforms import (
    RandomResizedCrop,
    Resize,
    CenterCrop,
    Compose,
    Normalize,
    ToTensor,
)

from transformers import (
    DefaultDataCollator,
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
)

In [ ]:
def get_image_size(image_processor):
    if "shortest_edge" in image_processor.size:
        return image_processor.size["shortest_edge"]
    return image_processor.size["height"]

def apply_transforms(examples, image_processor, is_train=True):
    image_size = get_image_size(image_processor)
    normalize = Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    )

    if is_train:
        transform = Compose([
            RandomResizedCrop(image_size),
            ToTensor(),
            normalize,
        ])
    else:
        transform = Compose([
            Resize(image_size),
            CenterCrop(image_size),
            ToTensor(),
            normalize,
        ])

    examples["pixel_values"] = [
        transform(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

In [ ]:
# -----------------------------------------------------
# Shared metric helpers for ConvNeXt evaluation
# -----------------------------------------------------
# Trainer needs compute_metrics(eval_pred), while later analysis needs
# probability, confidence, and top-k helper functions.

def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def top_k_accuracy_from_probs(probs, labels, k=5):
    """Computes top-k accuracy from [N, C] class probabilities."""
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)
    k = min(k, probs.shape[1])
    top_k_predictions = np.argpartition(probs, -k, axis=1)[:, -k:]
    return float(np.mean(np.any(top_k_predictions == labels[:, None], axis=1)))


def compute_metrics(eval_pred):
    """Hugging Face Trainer-compatible metric function."""
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    labels_np = eval_pred.label_ids.astype(int)
    probs = softmax_np(logits)
    preds = np.argmax(probs, axis=1)

    return {
        "accuracy": accuracy_score(labels_np, preds),
        "macro_f1": f1_score(labels_np, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_np, preds, average="weighted", zero_division=0),
        "top_5_accuracy": top_k_accuracy_from_probs(probs, labels_np, k=5),
    }


def get_topk_info(probs_row, id2label_mapping, k):
    """
    Returns top-k ids, dataset labels, natural labels, and probabilities
    for one sample.
    """
    probs_row = np.asarray(probs_row)
    k = min(k, len(probs_row))
    top_ids = np.argsort(probs_row)[::-1][:k].astype(int).tolist()
    top_dataset_labels = [id2label_mapping[i] for i in top_ids]
    top_natural_labels = [id2label_mapping[i].replace("_", " ") for i in top_ids]
    top_probs = [float(probs_row[i]) for i in top_ids]

    return top_ids, top_dataset_labels, top_natural_labels, top_probs

print("Metric helpers defined.")

In [ ]:
# -----------------------------
# Balanced Food101 subset config
# -----------------------------
DATASET_NAME = "ethz/food101"
NUM_SELECTED_CLASSES = 20
SAMPLES_PER_CLASS = 250
TEST_SIZE = 0.20
SEED = 42

# Load the full Food101 training split.
# We use the train split and create our own stratified train/validation split
# because this experiment is meant to compare fine-tuning strategies cheaply.
food_full_train = load_dataset(DATASET_NAME, split="train")
original_labels = food_full_train.features["label"].names

rng = random.Random(SEED)

# Group dataset indices by original Food101 class id
label_to_indices = defaultdict(list)
for idx, label_id in enumerate(food_full_train["label"]):
    label_to_indices[int(label_id)].append(idx)

# Keep only classes that have enough examples
eligible_label_ids = [
    label_id
    for label_id, indices in label_to_indices.items()
    if len(indices) >= SAMPLES_PER_CLASS
]

if len(eligible_label_ids) < NUM_SELECTED_CLASSES:
    raise ValueError(
        f"Only {len(eligible_label_ids)} classes have at least "
        f"{SAMPLES_PER_CLASS} samples. Need {NUM_SELECTED_CLASSES}."
    )

# Select 20 classes reproducibly
selected_old_label_ids = sorted(rng.sample(eligible_label_ids, NUM_SELECTED_CLASSES))

# Select exactly 250 images per selected class
selected_indices = []
for old_label_id in selected_old_label_ids:
    indices = label_to_indices[old_label_id].copy()
    rng.shuffle(indices)
    selected_indices.extend(indices[:SAMPLES_PER_CLASS])

rng.shuffle(selected_indices)

food_subset = food_full_train.select(selected_indices)

print(f"Total selected images: {len(food_subset)}")
print(f"Selected classes: {len(selected_old_label_ids)}")
print("Selected class names:")
for old_label_id in selected_old_label_ids:
    print(f"  old_id={old_label_id:3d} -> {original_labels[old_label_id]}")

# Stratified split while labels are still original Food101 label IDs
food = food_subset.train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
    stratify_by_column="label",
)

# Remap selected original labels to contiguous labels 0..19.
# This is important because the classifier head will have exactly 20 outputs.
old_to_new = {
    old_label_id: new_label_id
    for new_label_id, old_label_id in enumerate(selected_old_label_ids)
}

new_to_old = {
    new_label_id: old_label_id
    for old_label_id, new_label_id in old_to_new.items()
}

labels = [
    original_labels[new_to_old[new_label_id]]
    for new_label_id in range(NUM_SELECTED_CLASSES)
]

id2label = {
    new_label_id: label_name
    for new_label_id, label_name in enumerate(labels)
}

label2id = {
    label_name: new_label_id
    for new_label_id, label_name in id2label.items()
}

def remap_label(example):
    example["label"] = old_to_new[int(example["label"])]
    return example

food = DatasetDict({
    "train": food["train"].map(remap_label),
    "test": food["test"].map(remap_label),
})

# Sanity checks
train_counts = Counter(food["train"]["label"])
test_counts = Counter(food["test"]["label"])

print("\nAfter remapping:")
print(f"Train size: {len(food['train'])}")
print(f"Validation size: {len(food['test'])}")
print(f"Number of labels: {len(labels)}")
print(f"Train class counts: {sorted(train_counts.items())}")
print(f"Validation class counts: {sorted(test_counts.items())}")

assert len(food["train"]) + len(food["test"]) == NUM_SELECTED_CLASSES * SAMPLES_PER_CLASS
assert set(train_counts.keys()) == set(range(NUM_SELECTED_CLASSES))
assert set(test_counts.keys()) == set(range(NUM_SELECTED_CLASSES))

In [ ]:
print(f"Number of selected labels: {len(labels)}")
print("id2label:")
for class_id, class_name in id2label.items():
    print(f"  {class_id}: {class_name}")

print("\nlabel2id:")
for class_name, class_id in label2id.items():
    print(f"  {class_name}: {class_id}")

In [ ]:
# ConvNeXt-Tiny model from Hugging Face
convnext_model_id_hf = "facebook/convnext-tiny-224"

# Load the ConvNeXt-specific image processor
convnext_image_processor = AutoImageProcessor.from_pretrained(convnext_model_id_hf)

# Apply ConvNeXt-specific transforms to the already split balanced Food101 subset
food_convnext = food.copy()

food_convnext["train"] = food_convnext["train"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=True
    )
)

food_convnext["test"] = food_convnext["test"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=False
    )
)

In [ ]:
# Load the ConvNeXt-Tiny model with a new 20-class classification head
convnext_model_hf = AutoModelForImageClassification.from_pretrained(
    convnext_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Ensure all parameters require gradients for full fine-tuning
for param in convnext_model_hf.parameters():
    param.requires_grad = True

print("ConvNeXt-Tiny model loaded and configured for full fine-tuning.")
print(convnext_model_hf)

# Sanity checks
print("Number of labels:", convnext_model_hf.config.num_labels)
print("id2label length:", len(convnext_model_hf.config.id2label))
print("label2id length:", len(convnext_model_hf.config.label2id))

In [ ]:
data_collator = DefaultDataCollator()

# Define TrainingArguments for ConvNeXt-Tiny full fine-tuning
training_args_convnext_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_convnext_tiny_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_convnext_tiny_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False
)

# Initialize the Trainer for ConvNeXt-Tiny
trainer_convnext_hf = Trainer(
    model=convnext_model_hf,
    args=training_args_convnext_hf,
    data_collator=data_collator,
    train_dataset=food_convnext["train"],
    eval_dataset=food_convnext["test"],
    processing_class=convnext_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ConvNeXt-Tiny full model fine-tuning...")
train_results = trainer_convnext_hf.train()

In [ ]:
# -----------------------------------------------------
# ConvNeXt uncertainty analysis for VLM revalidation
# -----------------------------------------------------
# Goal:
#   Select samples using ONLY inference-time signals:
#       1. ConvNeXt top-1 confidence
#       2. ConvNeXt top1-top2 margin
#       3. ConvNeXt predictive entropy
#
# Important:
#   We do NOT select based on whether ConvNeXt was correct or incorrect.
#   In real inference, we do not know the true label.
#
# The VLM will reclassify every selected uncertain sample using only that
# sample's ConvNeXt top-k candidate labels.

print("Getting predictions from the trained ConvNeXt model...")
convnext_prediction_output = trainer_convnext_hf.predict(food_convnext["test"])

convnext_logits = convnext_prediction_output.predictions
if isinstance(convnext_logits, tuple):
    convnext_logits = convnext_logits[0]

true_labels = np.asarray(convnext_prediction_output.label_ids).astype(int)

convnext_probs = softmax_np(convnext_logits)
convnext_predictions = np.argmax(convnext_probs, axis=1).astype(int)

convnext_correct = convnext_predictions == true_labels
convnext_accuracy = float(np.mean(convnext_correct))
convnext_top5_accuracy = top_k_accuracy_from_probs(convnext_probs, true_labels, k=5)

convnext_top1_confidence = np.max(convnext_probs, axis=1)

sorted_probs = np.sort(convnext_probs, axis=1)
convnext_top2_confidence = (
    sorted_probs[:, -2]
    if convnext_probs.shape[1] > 1
    else np.zeros_like(convnext_top1_confidence)
)

convnext_margin = convnext_top1_confidence - convnext_top2_confidence

convnext_entropy = -np.sum(
    convnext_probs * np.log(np.clip(convnext_probs, 1e-12, 1.0)),
    axis=1,
)

print(f"ConvNeXt accuracy:       {convnext_accuracy:.4f}")
print(f"ConvNeXt top-5 accuracy: {convnext_top5_accuracy:.4f}")

# -----------------------------
# Routing settings
# -----------------------------
# These thresholds define when ConvNeXt is considered uncertain.
# Tune these values to control how many samples are sent to the VLM.
LOW_CONFIDENCE_THRESHOLD = 0.60
LOW_MARGIN_THRESHOLD = 0.20
HIGH_ENTROPY_THRESHOLD = 1.50

# Since we do not know correctness at inference time, use the same candidate
# count for every routed uncertain sample.
RERANK_CANDIDATE_K = 10

uncertain_mask = (
    (convnext_top1_confidence < LOW_CONFIDENCE_THRESHOLD)
    | (convnext_margin < LOW_MARGIN_THRESHOLD)
    | (convnext_entropy > HIGH_ENTROPY_THRESHOLD)
)

uncertain_indices = np.where(uncertain_mask)[0]

print(f"\nLow confidence threshold: {LOW_CONFIDENCE_THRESHOLD:.2f}")
print(f"Low margin threshold:     {LOW_MARGIN_THRESHOLD:.2f}")
print(f"High entropy threshold:   {HIGH_ENTROPY_THRESHOLD:.2f}")
print(f"Rerank candidate K:       {RERANK_CANDIDATE_K}")
print(f"Number of uncertain samples routed to VLM: {len(uncertain_indices)}")
print(f"Routing rate: {len(uncertain_indices) / len(true_labels):.4f}")

# These are evaluation-only diagnostics.
# They are NOT used for routing.
routed_originally_correct = int(convnext_correct[uncertain_indices].sum())
routed_originally_wrong = int((~convnext_correct[uncertain_indices]).sum())

print("\nEvaluation-only diagnostics:")
print(f"Routed samples originally correct: {routed_originally_correct}")
print(f"Routed samples originally wrong:   {routed_originally_wrong}")

# -----------------------------
# Build selected_cases_df
# -----------------------------
selected_rows = []

for sample_index in uncertain_indices:
    sample_index = int(sample_index)

    top_ids, top_dataset_labels, top_natural_labels, top_probs = get_topk_info(
        convnext_probs[sample_index],
        id2label,
        k=RERANK_CANDIDATE_K,
    )

    selected_rows.append({
        "sample_index": sample_index,
        "case_type": "uncertain",
        "candidate_k": RERANK_CANDIDATE_K,

        # Ground truth is kept only for evaluation after routing.
        "true_label_id": int(true_labels[sample_index]),
        "true_label": id2label[int(true_labels[sample_index])],

        # ConvNeXt prediction information.
        "convnext_pred_id": int(convnext_predictions[sample_index]),
        "convnext_pred_label": id2label[int(convnext_predictions[sample_index])],
        "convnext_correct": bool(convnext_correct[sample_index]),

        # Inference-time uncertainty signals.
        "convnext_top1_confidence": float(convnext_top1_confidence[sample_index]),
        "convnext_top2_confidence": float(convnext_top2_confidence[sample_index]),
        "convnext_margin": float(convnext_margin[sample_index]),
        "convnext_entropy": float(convnext_entropy[sample_index]),

        # Candidate list given to the VLM.
        "candidate_label_ids": top_ids,
        "candidate_dataset_labels": top_dataset_labels,
        "candidate_natural_labels": top_natural_labels,
        "candidate_confidences": top_probs,

        # Evaluation-only diagnostic.
        "true_label_in_candidates": int(true_labels[sample_index]) in top_ids,
    })

selected_cases_df = pd.DataFrame(selected_rows)

print("\nSelected case counts:")
display(selected_cases_df["case_type"].value_counts().to_frame("count"))

print("\nTrue label coverage inside routed candidate lists:")
display(
    selected_cases_df
    .groupby("case_type")["true_label_in_candidates"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "coverage_rate", "sum": "covered", "count": "total"})
    .round(4)
)

print("\nRouted uncertain samples: correct vs wrong before VLM:")
display(
    selected_cases_df["convnext_correct"]
    .value_counts()
    .rename(index={True: "originally_correct", False: "originally_wrong"})
    .to_frame("count")
)

print("\nPreview of routed uncertain cases for VLM revalidation:")
display(
    selected_cases_df[[
        "sample_index",
        "case_type",
        "candidate_k",
        "true_label",
        "convnext_pred_label",
        "convnext_correct",
        "convnext_top1_confidence",
        "convnext_margin",
        "convnext_entropy",
        "true_label_in_candidates",
        "candidate_dataset_labels",
        "candidate_confidences",
    ]].head(20)
)

In [ ]:
# SmolVLM model from Hugging Face.
# This is used as a zero-shot VLM classifier by scoring each candidate class label
# with image-conditioned log-likelihood, not by generating free-form text.
smolvlm_model_id = "HuggingFaceTB/SmolVLM-Instruct"

vlm_device = "cuda" if torch.cuda.is_available() else "cpu"
vlm_dtype = torch.float16 if vlm_device == "cuda" else torch.float32

processor_smolvlm = AutoProcessor.from_pretrained(smolvlm_model_id)
model_smolvlm = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id,
    torch_dtype=vlm_dtype,
)
model_smolvlm.to(vlm_device)
model_smolvlm.eval()

print(f"SmolVLM processor and model loaded on {vlm_device} with dtype={vlm_dtype}.")

In [ ]:
# -----------------------------------------------------
# VLM candidate-list revalidation helpers
# -----------------------------------------------------
# The VLM is NOT asked to classify from all 20 labels here.
# For each selected ConvNeXt case:
#   - low-confidence correct samples get the ConvNeXt top-5 candidates
#   - incorrect samples get the ConvNeXt top-10 candidates
# The VLM must choose exactly one label from that candidate list.


def vlm_label_name(label_name):
    return str(label_name).replace("_", " ")


def normalize_label_text(text):
    text = str(text).lower().strip()
    text = text.replace("_", " ")
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    prefixes = [
        "the answer is",
        "answer is",
        "answer",
        "the class is",
        "class is",
        "class",
        "the label is",
        "label is",
        "label",
        "the food is",
        "food is",
        "food",
        "this is",
        "it is",
        "it looks like",
    ]

    for prefix in prefixes:
        if text.startswith(prefix + " "):
            text = text[len(prefix):].strip()

    return text


def build_candidate_label_map(candidate_label_ids, id2label_mapping):
    """
    Builds a parser map restricted to the current candidate list.
    This prevents the VLM from being credited for labels outside the allowed list.
    """
    candidate_label_ids = [int(x) for x in candidate_label_ids]

    id_to_dataset_label = {
        label_id: id2label_mapping[label_id]
        for label_id in candidate_label_ids
    }

    id_to_natural_label = {
        label_id: vlm_label_name(id2label_mapping[label_id])
        for label_id in candidate_label_ids
    }

    normalized_to_id = {
        normalize_label_text(natural_label): label_id
        for label_id, natural_label in id_to_natural_label.items()
    }

    return {
        "candidate_label_ids": candidate_label_ids,
        "id_to_dataset_label": id_to_dataset_label,
        "id_to_natural_label": id_to_natural_label,
        "normalized_to_id": normalized_to_id,
        "candidate_natural_labels": [id_to_natural_label[i] for i in candidate_label_ids],
        "candidate_dataset_labels": [id_to_dataset_label[i] for i in candidate_label_ids],
    }


def parse_vlm_output_against_candidates(generated_text, candidate_label_ids, id2label_mapping):
    """
    Parses generated text into a class ID, but only if the generated label is
    one of the allowed candidate labels for that sample.
    """
    label_map = build_candidate_label_map(candidate_label_ids, id2label_mapping)
    normalized_output = normalize_label_text(generated_text)
    normalized_to_id = label_map["normalized_to_id"]

    # Exact match first.
    if normalized_output in normalized_to_id:
        pred_id = normalized_to_id[normalized_output]
        return pred_id, id2label_mapping[pred_id], normalized_output, True

    # Substring match, longest candidate first.
    candidates = sorted(
        normalized_to_id.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    )

    matches = []
    for normalized_label, label_id in candidates:
        pos = normalized_output.find(normalized_label)
        if pos != -1:
            matches.append((pos, -len(normalized_label), label_id))

    if matches:
        matches.sort()
        _, _, pred_id = matches[0]
        return pred_id, id2label_mapping[pred_id], normalized_output, True

    return -1, "INVALID_OUTPUT", normalized_output, False


def build_vlm_candidate_rerank_prompt(candidate_label_ids, id2label_mapping, processor=None):
    """
    Builds a prompt where the VLM must choose from only this sample's
    ConvNeXt candidate labels.
    """
    label_map = build_candidate_label_map(candidate_label_ids, id2label_mapping)
    candidate_lines = "\n".join([
        f"- {name}"
        for name in label_map["candidate_natural_labels"]
    ])

    user_text = (
        "A classifier was uncertain between these labels."
        "Use the image to choose the most visually likely class.\n"
        f"{candidate_lines}\n\n"
        "Answer with exactly one class name from the candidate list.\n"
        "Do not explain.\n"
        "Answer:"
    )

    if processor is not None and hasattr(processor, "apply_chat_template"):
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": user_text},
                    ],
                }
            ]
            return processor.apply_chat_template(messages, add_generation_prompt=True)
        except Exception as e:
            print("Chat template failed; falling back to manual <image> prompt.")
            print("Reason:", e)

    return "<image>\n" + user_text


def move_processor_inputs_to_device(inputs, device, dtype=None):
    moved = {}
    for key, value in inputs.items():
        value = value.to(device)
        if dtype is not None and torch.is_floating_point(value):
            value = value.to(dtype=dtype)
        moved[key] = value
    return moved


def generate_vlm_candidate_choice(
    model,
    processor,
    image,
    candidate_label_ids,
    id2label_mapping,
    device,
    dtype=None,
    max_new_tokens=12,
):
    """
    Runs one VLM generation call for one selected sample.
    The output is parsed only against the sample's allowed candidate labels.
    """
    image = image.convert("RGB")
    prompt = build_vlm_candidate_rerank_prompt(
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label_mapping,
        processor=processor,
    )

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    )
    inputs = move_processor_inputs_to_device(inputs, device=device, dtype=dtype)

    input_ids = inputs.get("input_ids", None)
    input_length = input_ids.shape[-1] if input_ids is not None else None

    tokenizer = getattr(processor, "tokenizer", None)
    eos_token_id = getattr(tokenizer, "eos_token_id", None) if tokenizer is not None else None
    pad_token_id = getattr(tokenizer, "pad_token_id", None) if tokenizer is not None else None
    if pad_token_id is None:
        pad_token_id = eos_token_id

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )

    if input_length is not None and generated_ids.shape[-1] > input_length:
        new_token_ids = generated_ids[:, input_length:]
    else:
        new_token_ids = generated_ids

    raw_output = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0].strip()

    pred_id, pred_label, normalized_output, valid_output = parse_vlm_output_against_candidates(
        generated_text=raw_output,
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label_mapping,
    )

    return {
        "vlm_pred_id": int(pred_id),
        "vlm_pred_label": pred_label,
        "vlm_valid_output": bool(valid_output),
        "vlm_raw_output": raw_output,
        "vlm_normalized_output": normalized_output,
    }

print("VLM candidate-list revalidation helpers defined.")

In [ ]:
# -----------------------------------------------------
# Run VLM revalidation on routed uncertain ConvNeXt cases
# -----------------------------------------------------
# This cell revalidates samples selected only by inference-time uncertainty:
#   - low top-1 confidence
#   - low top1-top2 margin
#   - high entropy
#
# The VLM receives the original image and the ConvNeXt top-k candidate labels.
#
# Evaluation-only questions:
#   1. How many originally wrong ConvNeXt predictions did the VLM fix?
#   2. How many originally correct ConvNeXt predictions did the VLM regress?
#   3. What is the net accuracy change on the full validation set?
#   4. How much latency was added?

assert "selected_cases_df" in globals(), "Run the ConvNeXt uncertainty-routing cell first."

print(f"Running VLM revalidation on {len(selected_cases_df)} routed uncertain cases...")
print(selected_cases_df["case_type"].value_counts())

model_smolvlm.eval()

rerank_rows = []
start_time = time.time()

for _, row in tqdm(
    selected_cases_df.iterrows(),
    total=len(selected_cases_df),
    desc="VLM revalidating uncertain cases",
):
    sample_index = int(row["sample_index"])

    # Use the raw image dataset, not the ConvNeXt-transformed dataset.
    # The VLM processor will apply its own image preprocessing.
    item = food["test"][sample_index]

    candidate_label_ids = [int(x) for x in row["candidate_label_ids"]]

    vlm_result = generate_vlm_candidate_choice(
        model=model_smolvlm,
        processor=processor_smolvlm,
        image=item["image"],
        candidate_label_ids=candidate_label_ids,
        id2label_mapping=id2label,
        device=vlm_device,
        dtype=vlm_dtype,
        max_new_tokens=12,
    )

    true_id = int(row["true_label_id"])
    convnext_pred_id = int(row["convnext_pred_id"])
    vlm_pred_id = int(vlm_result["vlm_pred_id"])

    convnext_correct_this_case = bool(row["convnext_correct"])
    vlm_correct = vlm_pred_id == true_id

    # Evaluation-only outcome flags.
    vlm_preserved_correct = convnext_correct_this_case and vlm_correct
    vlm_regressed_correct = convnext_correct_this_case and (not vlm_correct)

    vlm_fixed_convnext_error = (not convnext_correct_this_case) and vlm_correct

    vlm_kept_same_wrong_prediction = (
        (not convnext_correct_this_case)
        and vlm_result["vlm_valid_output"]
        and (vlm_pred_id == convnext_pred_id)
    )

    vlm_changed_to_different_wrong_prediction = (
        (not convnext_correct_this_case)
        and vlm_result["vlm_valid_output"]
        and (vlm_pred_id != convnext_pred_id)
        and (vlm_pred_id != true_id)
    )

    rerank_rows.append({
        **row.to_dict(),
        **vlm_result,
        "vlm_correct": bool(vlm_correct),
        "vlm_preserved_correct": bool(vlm_preserved_correct),
        "vlm_regressed_correct": bool(vlm_regressed_correct),
        "vlm_fixed_convnext_error": bool(vlm_fixed_convnext_error),
        "vlm_kept_same_wrong_prediction": bool(vlm_kept_same_wrong_prediction),
        "vlm_changed_to_different_wrong_prediction": bool(vlm_changed_to_different_wrong_prediction),
    })

vlm_revalidation_time = time.time() - start_time
vlm_revalidation_df = pd.DataFrame(rerank_rows)

print(f"\nVLM revalidation finished in {vlm_revalidation_time:.2f} seconds.")
print(
    "Average latency per routed sample: "
    f"{vlm_revalidation_time / max(len(vlm_revalidation_df), 1):.2f} seconds"
)

# -----------------------------
# Summary
# -----------------------------
routed_df = vlm_revalidation_df.copy()

originally_correct_df = routed_df[routed_df["convnext_correct"] == True]
originally_wrong_df = routed_df[routed_df["convnext_correct"] == False]

num_fixed = (
    int(originally_wrong_df["vlm_fixed_convnext_error"].sum())
    if len(originally_wrong_df)
    else 0
)

num_regressed = (
    int(originally_correct_df["vlm_regressed_correct"].sum())
    if len(originally_correct_df)
    else 0
)

net_change = num_fixed - num_regressed

convnext_correct_total = int(convnext_correct.sum())
hybrid_correct_total = convnext_correct_total + net_change
hybrid_accuracy = hybrid_correct_total / len(true_labels)

summary = {
    "total_dataset_samples": len(true_labels),

    "convnext_correct_total": convnext_correct_total,
    "convnext_accuracy": convnext_accuracy,

    "routed_uncertain_cases": len(routed_df),
    "routing_rate": len(routed_df) / len(true_labels),

    "routed_originally_correct_cases": len(originally_correct_df),
    "routed_originally_wrong_cases": len(originally_wrong_df),

    "vlm_invalid_outputs": int((~routed_df["vlm_valid_output"]).sum()) if len(routed_df) else 0,

    "originally_correct_preserved_by_vlm": (
        int(originally_correct_df["vlm_preserved_correct"].sum())
        if len(originally_correct_df)
        else 0
    ),
    "originally_correct_regressed_by_vlm": num_regressed,

    "originally_wrong_fixed_by_vlm": num_fixed,
    "originally_wrong_same_wrong_by_vlm": (
        int(originally_wrong_df["vlm_kept_same_wrong_prediction"].sum())
        if len(originally_wrong_df)
        else 0
    ),
    "originally_wrong_different_wrong_by_vlm": (
        int(originally_wrong_df["vlm_changed_to_different_wrong_prediction"].sum())
        if len(originally_wrong_df)
        else 0
    ),

    "true_label_candidate_coverage": (
        float(routed_df["true_label_in_candidates"].mean())
        if len(routed_df)
        else np.nan
    ),

    "net_change_on_routed_cases": net_change,

    "hybrid_correct_total_after_vlm_override": hybrid_correct_total,
    "hybrid_accuracy_after_vlm_override": hybrid_accuracy,
    "hybrid_accuracy_gain": hybrid_accuracy - convnext_accuracy,

    "vlm_revalidation_time_sec": vlm_revalidation_time,
    "avg_latency_per_routed_sample_sec": (
        vlm_revalidation_time / max(len(routed_df), 1)
    ),
    "avg_added_latency_per_dataset_sample_sec": (
        vlm_revalidation_time / max(len(true_labels), 1)
    ),
}

summary_df = pd.DataFrame([summary]).T.rename(columns={0: "value"})

print("\n--- VLM Revalidation Summary for Inference-Time Uncertain Cases ---")
display(summary_df)

print("\n--- Routed uncertain cases: originally correct vs originally wrong ---")
display(
    routed_df
    .groupby("convnext_correct")[[
        "true_label_in_candidates",
        "vlm_valid_output",
        "vlm_correct",
    ]]
    .mean()
    .rename(index={True: "originally_correct", False: "originally_wrong"})
    .round(4)
)

print("\n--- VLM fixed originally wrong ConvNeXt predictions ---")
fixed_df = routed_df[routed_df["vlm_fixed_convnext_error"] == True]
if len(fixed_df) > 0:
    display(fixed_df[[
        "sample_index",
        "true_label",
        "convnext_pred_label",
        "vlm_pred_label",
        "convnext_top1_confidence",
        "convnext_margin",
        "convnext_entropy",
        "candidate_dataset_labels",
        "candidate_confidences",
        "vlm_raw_output",
    ]].head(30))
else:
    print("No originally wrong ConvNeXt predictions were fixed by the VLM.")

print("\n--- VLM regressed originally correct ConvNeXt predictions ---")
regressed_df = routed_df[routed_df["vlm_regressed_correct"] == True]
if len(regressed_df) > 0:
    display(regressed_df[[
        "sample_index",
        "true_label",
        "convnext_pred_label",
        "vlm_pred_label",
        "convnext_top1_confidence",
        "convnext_margin",
        "convnext_entropy",
        "candidate_dataset_labels",
        "candidate_confidences",
        "vlm_raw_output",
    ]].head(30))
else:
    print("No originally correct ConvNeXt predictions were regressed by the VLM.")

print("\n--- Preview of all VLM revalidation results ---")
display(routed_df[[
    "sample_index",
    "case_type",
    "candidate_k",
    "true_label",
    "convnext_pred_label",
    "vlm_pred_label",
    "convnext_correct",
    "convnext_top1_confidence",
    "convnext_margin",
    "convnext_entropy",
    "true_label_in_candidates",
    "vlm_valid_output",
    "vlm_correct",
    "vlm_fixed_convnext_error",
    "vlm_regressed_correct",
    "candidate_dataset_labels",
    "candidate_confidences",
    "vlm_raw_output",
]].head(50))